# Base Student Output Generation - Google Colab (Checkpoint-Based)
Run on T4 GPU. Generates base student outputs **one checkpoint at a time** (5K records each).
Set `CHECKPOINT` in Cell 2 to control which checkpoint to process.

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth -q
!pip install supabase python-dotenv -q

In [ ]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════
# SET THESE VALUES:
CHECKPOINT = 2        # Which checkpoint to generate (1-10)
SUPABASE_URL = ""     # Your Supabase URL
SUPABASE_KEY = ""     # Your Supabase service role key
# ═══════════════════════════════════════════════════

MAX_NEW_TOKENS = 256
BATCH_SIZE = 100
RECORDS_PER_CHECKPOINT = 5000

In [ ]:
# Cell 3: Load model (4-bit quantized for speed)
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True

from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print("Loading model...")
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded on {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Cell 4: Connect to Supabase and check checkpoint progress
from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Count remaining for THIS checkpoint
pending = supabase.table("modelcomp_50k").select("id", count="exact") \
    .eq("checkpoint", CHECKPOINT).is_("student_output", "null").execute()
done = supabase.table("modelcomp_50k").select("id", count="exact") \
    .eq("checkpoint", CHECKPOINT).not_.is_("student_output", "null").execute()

print(f"{'='*50}")
print(f"📋 CHECKPOINT {CHECKPOINT} STATUS")
print(f"{'='*50}")
print(f"   Done:      {done.count:,} / {RECORDS_PER_CHECKPOINT:,}")
print(f"   Pending:   {pending.count:,}")
print(f"   Progress:  {done.count/RECORDS_PER_CHECKPOINT*100:.1f}%")
print(f"{'='*50}")

In [ ]:
# Cell 5: Generation function
import time

MAX_NEW_TOKENS = 256

def generate_output(instruction: str, context: str = "") -> tuple:
    """Generate output and return (text, latency)"""
    start_time = time.time()
    
    # Build prompt
    if context:
        user_content = f"Context: {context}\n\nInstruction: {instruction}"
    else:
        user_content = instruction
    
    messages = [
        {"role": "user", "content": user_content}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the new tokens
    new_tokens = outputs[0][inputs.shape[1]:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    latency = time.time() - start_time
    return output_text.strip(), latency

In [ ]:
# Cell 6: Test generation (run this first to verify it works)
test_output, test_latency = generate_output("What is machine learning?", "")
print(f"Test output ({test_latency:.2f}s):\n{test_output[:200]}...")

In [ ]:
# Cell 7: Main generation loop - processes remaining records for THIS CHECKPOINT
from tqdm import tqdm

total_processed = 0
errors = []

print(f"Starting generation for Checkpoint {CHECKPOINT}...")
print("Progress saves after EACH record - safe to stop anytime!")
print("-" * 50)

while True:
    # Fetch batch of records without student_output FOR THIS CHECKPOINT
    result = supabase.table("modelcomp_50k") \
        .select("id, input, context") \
        .eq("checkpoint", CHECKPOINT) \
        .is_("student_output", "null") \
        .order("id") \
        .limit(BATCH_SIZE) \
        .execute()
    
    records = result.data
    if not records:
        print(f"\n✅ Checkpoint {CHECKPOINT} complete!")
        break
    
    print(f"\nProcessing batch of {len(records)} records (Checkpoint {CHECKPOINT})...")
    
    for record in tqdm(records, desc=f"Ckpt {CHECKPOINT}"):
        try:
            record_id = record["id"]
            instruction = record["input"] or ""
            context = record["context"] or ""
            
            # Generate
            output, latency = generate_output(instruction, context)
            
            # Save immediately to Supabase
            supabase.table("modelcomp_50k").update({
                "student_output": output,
                "generation_latency": round(latency, 3)
            }).eq("id", record_id).execute()
            
            total_processed += 1
            
        except Exception as e:
            errors.append({"id": record_id, "error": str(e)})
            print(f"\nError on {record_id}: {e}")
    
    # Progress update
    remaining_result = supabase.table("modelcomp_50k").select("id", count="exact") \
        .eq("checkpoint", CHECKPOINT).is_("student_output", "null").execute()
    done_now = RECORDS_PER_CHECKPOINT - remaining_result.count
    print(f"Ckpt {CHECKPOINT}: {done_now}/{RECORDS_PER_CHECKPOINT} | This session: {total_processed} | Errors: {len(errors)}")

print(f"\n{'='*50}")
print(f"Checkpoint {CHECKPOINT} - Total processed this session: {total_processed}")
print(f"Errors: {len(errors)}")
if total_processed > 0:
    print(f"\n🎯 Next: Run 12_train_incremental.py --checkpoint {CHECKPOINT} --init")
    print(f"   Then: Change CHECKPOINT = {CHECKPOINT + 1} and re-run this notebook")

In [ ]:
# Cell 8: Check progress across all checkpoints
print(f"{'='*60}")
print(f"📋 ALL CHECKPOINTS STATUS")
print(f"{'='*60}")
for ckpt in range(1, 11):
    done = supabase.table("modelcomp_50k").select("id", count="exact") \
        .eq("checkpoint", ckpt).not_.is_("student_output", "null").execute()
    pct = done.count / RECORDS_PER_CHECKPOINT * 100
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    status = "✅" if done.count >= RECORDS_PER_CHECKPOINT else "⏳"
    print(f"   {status} Ckpt {ckpt:2d}: {done.count:5,}/{RECORDS_PER_CHECKPOINT:,} [{bar}] {pct:.0f}%")
print(f"{'='*60}")